# NormalObjects - Creative Complaint Handler (LangChain) Workflow

This notebook is the lab workflow version of `normalobjects_langchain.py`.

Run it in VS Code with the project virtual environment selected as the notebook kernel. The notebook demonstrates setup checks, tool registration, agent execution, and tool usage analysis.

## 1. Environment check

Before running the real agent, make sure:

- You created `.venv`
- You installed `requirements.txt`
- You copied `.env.example` to `.env`
- Your `.env` contains `OPENAI_API_KEY=...`

The first code cell confirms the notebook can import the project script.

In [1]:
from pathlib import Path
import os
import sys
from dotenv import load_dotenv

# Make the repository root importable when this notebook is inside notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
env_path = PROJECT_ROOT / '.env'
print(".env exists:", env_path.exists())

# Load .env (if present) and validate OPENAI_API_KEY
if env_path.exists():
    load_dotenv(env_path)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
if OPENAI_API_KEY:
    if 'sk-your' in OPENAI_API_KEY or (OPENAI_API_KEY.startswith('sk-') and 'REPLACE' in OPENAI_API_KEY.upper()):
        print("WARNING: OPENAI_API_KEY looks like a placeholder. Replace it with your real key.")
    else:
        print("OPENAI_API_KEY found (not displayed).")
else:
    print("OPENAI_API_KEY not set. Copy .env.example -> .env and add your key or set env var.")

Project root: /Users/mymacbook/Downloads/normalobjects-creative-complaint-handler
Python executable: /Users/mymacbook/Downloads/normalobjects-creative-complaint-handler/.venv/bin/python
.env exists: True
OPENAI_API_KEY found (not displayed).


In [2]:
%pip install python-dotenv langchain langchain-openai openai ipykernel

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys

print(sys.executable)
print(sys.version)

/Users/mymacbook/Downloads/normalobjects-creative-complaint-handler/.venv/bin/python
3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]


## 2. Import the lab solution

This imports the main functions and custom tools from `normalobjects_langchain.py`.

In [4]:
import sys
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from normalobjects_langchain import (
    TOOLS,
    SAMPLE_COMPLAINTS,
    build_agent_executor,
    handle_complaint,
    summarize_tool_usage,
    save_demo_markdown,
    run_mock_demo,
)

print("Lab solution imported successfully.")
print(f"Python: {sys.executable}")
print(f"Project root: {project_root}")
print(f"Loaded {len(TOOLS)} tools:")

for tool in TOOLS:
    print(f"- {tool.name}: {tool.description.splitlines()[0]}")

Lab solution imported successfully.
Python: /Users/mymacbook/Downloads/normalobjects-creative-complaint-handler/.venv/bin/python
Project root: /Users/mymacbook/Downloads/normalobjects-creative-complaint-handler
Loaded 5 tools:
- consult_demogorgon: Get the Demogorgon's perspective on a complaint about the Downside Up.
- check_hawkins_records: Search fictional Hawkins/Walvins anomaly records for clues.
- cast_interdimensional_spell: Suggest creative ritual-style fixes for a Downside Up complaint.
- gather_party_wisdom: Ask the Bureau's kid-investigator party for collective wisdom.
- consult_eleven_signal: Interpret psychic/static signals related to a complaint.


## 3. Optional no-API mock check

Run this cell first if you only want to verify the project works before using API tokens. This does **not** replace the real LangChain agent run for final submission.

In [6]:
mock_result = run_mock_demo()
mock_result["stats"]

Mock demo completed. This confirms files and environment structure, not API connectivity.
Demo evidence saved to: outputs/demo_results.md
Tool counts: {'check_hawkins_records': 3, 'gather_party_wisdom': 3, 'cast_interdimensional_spell': 3, 'consult_demogorgon': 2, 'consult_eleven_signal': 1}


{'total_tool_calls': 12,
 'tool_counts': {'check_hawkins_records': 3,
  'gather_party_wisdom': 3,
  'cast_interdimensional_spell': 3,
  'consult_demogorgon': 2,
  'consult_eleven_signal': 1},
 'most_used': 'check_hawkins_records',
 'tool_sequences': [['check_hawkins_records',
   'gather_party_wisdom',
   'cast_interdimensional_spell'],
  ['check_hawkins_records',
   'consult_demogorgon',
   'cast_interdimensional_spell'],
  ['consult_eleven_signal',
   'gather_party_wisdom',
   'cast_interdimensional_spell'],
  ['check_hawkins_records', 'gather_party_wisdom', 'consult_demogorgon']]}

In [7]:
from dotenv import load_dotenv
import os

load_dotenv(project_root / ".env", override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found. Check your .env file.")

if "your" in api_key.lower() or "here" in api_key.lower():
    raise ValueError("OPENAI_API_KEY is still a placeholder. Replace it with your real API key.")

print("API key loaded.")
print(f"Using model: {os.getenv('OPENAI_MODEL', 'gpt-4o-mini')}")

API key loaded.
Using model: gpt-4o-mini


## 4. Build the real LangChain agent

Run this cell after your `.env` file contains `OPENAI_API_KEY`.

The lab uses `AgentExecutor`, `create_openai_tools_agent`, `ChatPromptTemplate`, `MessagesPlaceholder`, and custom `@tool` functions, matching the lab template.

In [8]:
# Run this only after setting OPENAI_API_KEY in .env.
agent_executor = build_agent_executor(verbose=True)
print("Agent executor created successfully.")

Agent executor created successfully.


In [9]:
agent_executor = build_agent_executor(verbose=True)
print("Agent executor created successfully.")

Agent executor created successfully.


## 5. Handle at least three complaints

The lab asks for a demonstration showing at least 3 different complaints handled, tool usage patterns, and creative solutions. This cell handles four.

In [11]:
results = []

three_complaints = SAMPLE_COMPLAINTS[:3]

for complaint in three_complaints:
    item = handle_complaint(agent_executor, complaint)
    results.append(item)

    print("\n" + "=" * 80)
    print("COMPLAINT:")
    print(item["complaint"])

    print("\nTOOL SEQUENCE:")
    if item["tool_sequence"]:
        print(" -> ".join(item["tool_sequence"]))
    else:
        print("No tools used")

    print("\nRESPONSE:")
    print(item["response"])



> Entering new AgentExecutor chain...

Invoking: `consult_demogorgon` with `{'complaint': 'Why do demogorgons sometimes eat people and sometimes just stare at lamps?'}`


The Demogorgon scratches a symbol into the wall about: 'Why do demogorgons sometimes eat people and sometimes just stare at lamps?'. It seems to say consistency is a human superstition, but patterns appear near magnets, music, and unattended snack food.
Invoking: `gather_party_wisdom` with `{'question': 'Why do demogorgons exhibit such contrasting behaviors?'}`


Lucas: 'Creatures are territorial and opportunistic.' Dustin: 'Fear, food, and electrical interference could explain the inconsistent behavior.' Mike: 'So the answer is not random; it is badly documented.' Will: 'They also react to attention. Try not to be the loudest snack.'Ah, the perplexing behavior of demogorgons! 🦖 Let's dive into the delightful chaos of their actions.

From the Demogorgon's perspective, it seems to suggest that "consistency is a human

## 6. Analyze tool usage patterns

This shows how the agent chained tools in flexible orders. The result supports the required short reflection in `lab_summary.md`.

In [12]:
stats = summarize_tool_usage(results)

print("Total tool calls:", stats["total_tool_calls"])
print("Tool counts:", stats["tool_counts"])
print("Most used tool:", stats["most_used"])

print("\nTool sequences:")
for index, sequence in enumerate(stats["tool_sequences"], start=1):
    print(f"{index}. {' -> '.join(sequence) or 'No tools used'}")

Total tool calls: 6
Tool counts: {'consult_demogorgon': 2, 'gather_party_wisdom': 2, 'check_hawkins_records': 1, 'consult_eleven_signal': 1}
Most used tool: consult_demogorgon

Tool sequences:
1. consult_demogorgon -> gather_party_wisdom
2. check_hawkins_records -> consult_demogorgon
3. consult_eleven_signal -> gather_party_wisdom


## 7. Save demo evidence

This writes `outputs/demo_results.md`, which you can commit to GitHub as the demonstration evidence.

In [14]:
output_path = save_demo_markdown(results, stats)
print("Saved:", output_path)

Saved: outputs/demo_results.md


## 8. Final submission reminder

Submit the URL of the GitHub repository. Do not upload only a cloud notebook link.

Make sure these files exist at the repository root:

- `normalobjects_langchain.py`
- `README.md`
- `lab_summary.md`
- `requirements.txt`

Also include:

- `notebooks/normalobjects_workflow.ipynb`
- `outputs/demo_results.md`

Never commit `.env`.